### Experiment tracking
Because the only difference between science and messing around is writing it down.

In [ ]:
# Importing packages
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn import metrics
import pickle

In [ ]:
df = pd.read_csv("diabetes.csv")
columns = df.columns
for column in columns:
    df[column] = df[column].astype(int)
df.rename(columns={'Diabetes_012': 'class'}, inplace=True)


In [ ]:
# Let's shuffle the train/ test data split
X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis=1),df['class'], test_size=0.25, random_state=42)
model_1 = LogisticRegression(max_iter=5, random_state=42)
model_1.fit(X_train, y_train)

In [ ]:
# Scoring
train_score = model_1.score(X_train, y_train)
print(f"Training Accuracy: {round(train_score*100)}%")
test_score = model_1.score(X_test, y_test)
print(f"Testing Accuracy: {round(test_score*100)}%")
# Predictions for X_test
y_pred = model_1.predict(X_test)
# F1 Score
f1_value = f1_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"F1 Score: {round(f1_value,2)*100}")
# Recall score
recall_value = recall_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"Recall Score: {round(recall_value,2)*100}")
# Precision Score
precision_value = precision_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"Precision Score: {round(precision_value,2)*100}")
#define metrics
y_pred_proba = model_1.predict_proba(X_test)[::,1]
# auc = metrics.roc_auc_score(y_test, y_pred_proba)
# print(f"AUC Score: {round(auc,2)}")
cm = confusion_matrix(y_test, y_pred, labels=model_1.classes_)
tf_pf = cm[0][0]
print(f"Amount Predicted False where Actual False: {tf_pf}")
tf_pt = cm[0][1]
print(f"Amount Predicted True where Actual False: {tf_pt}")
tt_pf = cm[1][0]
print(f"Amount Predicted False where Actual True: {tt_pf}")
tt_pt = cm[1][1]
print(f"Amount Predicted True where Actual True: {tt_pt}")

In [ ]:
# Save model
with open('model_1.pkl', 'wb') as file:
    pickle.dump(model_1, file)

In [ ]:
model_2 = LogisticRegression(max_iter=1000, random_state=42, solver='saga', penalty='l2', class_weight='balanced')
model_2.fit(X_train, y_train)

In [ ]:
# Scoring
train_score = model_2.score(X_train, y_train)
print(f"Training Accuracy: {round(train_score*100)}%")
test_score = model_2.score(X_test, y_test)
print(f"Testing Accuracy: {round(test_score*100)}%")
# Predictions for X_test
y_pred = model_2.predict(X_test)
# F1 Score
f1_value = f1_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"F1 Score: {round(f1_value,2)*100}")
# Recall score
recall_value = recall_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"Recall Score: {round(recall_value,2)*100}")
# Precision Score
precision_value = precision_score(y_true=y_test, y_pred=y_pred, average='weighted')
print(f"Precision Score: {round(precision_value,2)*100}")
#define metrics
y_pred_proba = model_1.predict_proba(X_test)[::,1]
# auc = metrics.roc_auc_score(y_test, y_pred_proba)
# print(f"AUC Score: {round(auc,2)}")
cm = confusion_matrix(y_test, y_pred, labels=model_1.classes_)
tf_pf = cm[0][0]
print(f"Amount Predicted False where Actual False: {tf_pf}")
tf_pt = cm[0][1]
print(f"Amount Predicted True where Actual False: {tf_pt}")
tt_pf = cm[1][0]
print(f"Amount Predicted False where Actual True: {tt_pf}")
tt_pt = cm[1][1]
print(f"Amount Predicted True where Actual True: {tt_pt}")

In [ ]:
# Save model
with open('model_2.pkl', 'wb') as file:
    pickle.dump(model_2, file)

## Introducing MLflow for experiment tracking

To get started, install MLflow via pip: \
`pip install mlflow` \
After installation, you can launch the MLflow UI with the command: \
`mlflow ui`

In [ ]:
import mlflow
import mlflow.sklearn

In [ ]:

import os

# Pin everything to the notebook's own folder so this runs on any machine.
# In a Jupyter kernel, the working directory is the notebook's directory.
BASE_DIR = os.getcwd()
DB_PATH = os.path.join(BASE_DIR, "mlflow.db")
ARTIFACT_DIR = os.path.join(BASE_DIR, "mlruns")

mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# Create the experiment with an explicit artifact location the first time,
# so artifacts are written next to this notebook (not /home/davidjeon).
if mlflow.get_experiment_by_name("diabetes_classification") is None:
    mlflow.create_experiment(
        "diabetes_classification",
        artifact_location=f"file://{ARTIFACT_DIR}",
    )
mlflow.set_experiment("diabetes_classification")

# Let's shuffle the train/ test data split
X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis=1),df['class'], test_size=0.25, random_state=42)

# Start an MLflow experiment
with mlflow.start_run():
    model_3 = LogisticRegression(max_iter=1000, random_state=42)
    model_3.fit(X_train, y_train)
    predictions = model_3.predict(X_test)

    # Log model parameters
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("max_iter", model_3.max_iter)
    mlflow.log_param("solver", model_3.solver)
    mlflow.log_param("test_size", 0.25)

    # Log performance metrics
    mlflow.log_metric("accuracy", model_3.score(X_test, y_test))
    mlflow.log_metric("train_accuracy", model_3.score(X_train, y_train))
    mlflow.log_metric("f1_score", f1_score(y_true=y_test, y_pred=predictions, average='weighted'))
    mlflow.log_metric("precision", precision_score(y_true=y_test, y_pred=predictions, average='weighted'))
    mlflow.log_metric("recall", recall_score(y_true=y_test, y_pred=predictions, average='weighted'))

    # Log the model itself
    mlflow.sklearn.log_model(model_3, "logistic_regression_model")
    print(f"Model logged with F1 Score: {f1_score(y_true=y_test, y_pred=predictions, average='weighted'):.4f}")
    print(f"Accuracy: {model_3.score(X_test, y_test):.4f}")
    print(f"Precision: {precision_score(y_true=y_test, y_pred=predictions, average='weighted'):.4f}")
    print(f"Recall: {recall_score(y_true=y_test, y_pred=predictions, average='weighted'):.4f}")


In [ ]:
# To view the MLflow UI, navigate to the directory where this notebook is located and run the following command in your terminal:
# mlflow ui --backend-store-uri sqlite:///mlflow.db --default-artifact-root file